# Lab | Tools prompting


## Intro

In this lab you'll teach a chat model that has **no native tool-calling support** how to call tools anyway — just by prompting it correctly.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: defining `add` and `multiply` tools, writing a prompt that describes them to the model, parsing the model's JSON tool-call output, and invoking the right tool with the right arguments.
2. **Replicate it yourself with new tools** — Then, you'll replace the two demo tools with **3 new tools** of your own choosing and adjust the prompts accordingly, rebuilding the same pipeline so the model can correctly select and invoke your new tools.

By the end of this lab, you'll understand how ad-hoc tool calling works under the hood and be able to apply the pattern to any tools you design.

<br>

## How to add ad-hoc tool calling capability to LLMs and Chat Models

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

<br>

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

<br>

## Initial Setup


You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

> **Running this in Google Colab?** Unlike the other labs, this one does **not** need an API key from Colab Secrets — `phi3` runs locally via Ollama, with no account or key involved. Colab just needs Ollama installed and its server started as a background process first (a personal machine running the desktop Ollama app already has this covered). Run the cell below once per Colab session before creating the `Ollama` model below.

In [ ]:
# Colab-only setup: install Ollama and start its server in the background,
# then pull the phi3 model (a few GB download, only needed once per session).
# Skip this cell entirely if you're running locally with Ollama already installed
# and `ollama run phi3` working from a terminal - just run the cells below instead.
import subprocess
import time

# Install the Ollama CLI/server.
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Start the Ollama server in the background (it needs to keep running while this
# notebook is using it).
server_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give the server a moment to come up

# Download the phi3 model (only needs to happen once per Colab session).
subprocess.run(["ollama", "pull", "phi3"], check=True)
print("Ollama server running and phi3 model ready.")

In [ ]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

<br>

##  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

### Windows

1. **Download Ollama for Windows**
   Go to: [https://ollama.com/download](https://ollama.com/download)
   Download and run the installer.

2. **Verify Installation**
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  macOS

1. **Install via Homebrew**
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Linux

1. **Install Ollama**
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you're connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


<br>

## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [ ]:
from langchain_core.tools import tool


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def add(x: int, y: int) -> int:
    "Add two numbers."
    return x + y


tools = [multiply, add]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
multiply
Multiply two numbers together.
{'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}
--
add
Add two numbers.
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


In [ ]:
multiply.invoke({"x": 4, "y": 5})

20.0

<br>

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


In [ ]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [ ]:
chain = prompt | model
message = chain.invoke({"input": "what's 3 plus 1132"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "add",
  "arguments": {
    "x": 3,
    "y": 1132
  }
}
```

I used the `add` tool because you're asking for an addition operation. The arguments dictionary includes 'x' with value 3 and 'y' with value 1132 as per your question to add these two numbers together.


<br>

## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's thirteen times 4"})

{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}

<br>

Amazing! 🎉

We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!


<br>

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [ ]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

<br>

Let's test this out 🧪!

In [ ]:
invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

<br>

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [ ]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's thirteen times 4.14137281"})

53.83784653

<br>

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's thirteen times 4.14137281"})

{'name': 'multiply',\n 'arguments': {'x': 13, 'y': 4.14137281},\n 'output': 53.83784653}

<br>
<hr>
<br>
<br>

## 🚀 Your turn

Time to make this lab your own!

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information. In reality, if you're using more complex tools, you may start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

Replace the `multiply` and `add` tools with **3 new tools** of your choosing. They don't have to be math-related — a temperature converter, a string reverser, a word counter, a currency converter... pick whatever sounds fun to build.

<br>


Here's what to do:

1. **Design your 3 tools** — Write 3 new functions decorated with `@tool`, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.
2. **Update the tools list** — Replace `tools = [multiply, add]` with your 3 new tools.
3. **Regenerate the prompt** — Re-run `render_text_description(tools)` and rebuild `system_prompt`/`prompt` so the model sees the correct tool names, descriptions, and arguments.
4. **Rebuild the chain** — Recreate `chain = prompt | model | JsonOutputParser() | invoke_tool` with your new tools (the `invoke_tool` function already looks tools up dynamically, so it should work as-is).
5. **Put it to the test** — Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.
6. **Reflect** — In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

<br>


💡 **Tip:**

- Print `rendered_tools` after updating your tools list to double check the model is actually seeing the descriptions you expect.

<br>


⭐️ **Bonus ideas:**

- Add a 4th tool that takes more than 2 arguments, or a mix of types (e.g. `str` and `int`), and see if the model still handles it correctly.
- Add a few-shot example to your `system_prompt`: include one sample question together with the exact JSON output you'd want the model to produce for it. This gives the model a concrete pattern to copy, which often makes its answers more consistent.
- Add error handling around `invoke_tool`: if the tool name doesn't exist or the arguments are wrong, catch that error instead of crashing, then send the error message back to the model and ask it to try again with a corrected JSON blob.
- Ask a question that doesn't match any tool at all - how does the model respond, and how could you handle that gracefully in `invoke_tool`?


## My 3 New Tools

I picked three non-math, everyday-utility tools: a **temperature converter** (Celsius <-> Fahrenheit), a **string reverser**, and a **word counter**. As a bonus, I also added a 4th tool with 3 mixed-type arguments (a text repeater with a separator) to test whether `phi3` still handles a slightly more complex signature correctly.

In [ ]:
from langchain_core.tools import tool


@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


@tool
def reverse_string(text: str) -> str:
    """Reverse the characters in a string."""
    return text[::-1]


@tool
def count_words(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())


@tool
def repeat_text(text: str, times: int, separator: str = " ") -> str:
    """Repeat a piece of text a given number of times, joined by a separator.
    Bonus tool with 3 mixed-type arguments (str, int, str) to test a more
    complex tool signature."""
    return separator.join([text] * times)


tools = [celsius_to_fahrenheit, reverse_string, count_words, repeat_text]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

In [ ]:
rendered_tools = render_text_description(tools)
print(rendered_tools)

In [ ]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.

Here is an example. If the user asks: "how many words are in the sentence the quick brown fox",
you should respond with exactly:
{{"name": "count_words", "arguments": {{"text": "the quick brown fox"}}}}
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [ ]:
# invoke_tool already looks tools up dynamically from the `tools` list defined
# above, so it works as-is with our new tools - no changes needed there.
chain = prompt | model | JsonOutputParser() | invoke_tool

In [ ]:
# One test question per tool.
print(chain.invoke({"input": "What is 100 degrees Celsius in Fahrenheit?"}))
print(chain.invoke({"input": "Reverse the word langchain"}))
print(chain.invoke({"input": "How many words are in the sentence: the quick brown fox jumps over the lazy dog"}))
print(chain.invoke({"input": "Repeat the word hello 3 times separated by a dash"}))

In [ ]:
# A question that doesn't match any tool at all, to see how the model responds.
chain_json_only = prompt | model | JsonOutputParser()
print(chain_json_only.invoke({"input": "What's the capital of France?"}))

## Reflection

Since I don't have Ollama/`phi3` running in this environment, I couldn't execute the cells above myself - they're ready to run once you have Ollama + phi3 available (either locally, or via the Colab setup cell near the top of this notebook). Based on the class demo's behavior with the original `add`/`multiply` tools, here's what I'd expect and what to watch for when you run these:

- **Temperature converter and word counter** are the most likely to work reliably first try: their inputs map almost word-for-word onto the user's phrasing ("100 degrees Celsius" -> `celsius=100`, "how many words... the quick brown fox..." -> `text="the quick brown fox..."`), similar to how the demo's `add`/`multiply` calls cleanly extracted `x`/`y` from "what's 3 plus 1132".
- **String reverser** is a good test of whether the model correctly identifies *which word* is the argument vs. part of the instruction - e.g. for "Reverse the word langchain", the risk is the model passing `"Reverse the word langchain"` as the whole `text` argument instead of just `"langchain"`. This is the same class of small-format mistake as when a model wraps its JSON in extra prose (as `phi3` did in the class demo's very first response, adding an explanation *after* the JSON block).
- **`repeat_text` (bonus, 3 mixed-type args)** is the one most likely to expose real weaknesses: a `str`+`int`+`str` signature gives the model more chances to mislabel an argument (e.g. treating `"dash"` as a literal separator string instead of substituting `"-"`, or leaving `separator` at its default instead of picking up "separated by a dash"). Smaller/non-tool-tuned models like `phi3` tend to do noticeably worse as argument count and type diversity increase - this mirrors the general pattern from the earlier PAL/ReAct quiz questions in this project, where models are more reliable on single, clearly-scoped instructions than on ones requiring several coordinated details at once.
- **The "capital of France" trap question** is expected to be the most fragile case: since `invoke_tool` has no fallback for "no tool applies," if the model tries to force the question into one of the 4 tools anyway (rather than declining), `invoke_tool` will either run a nonsensical tool call or KeyError on an invalid tool name. This is a good candidate for the bonus error-handling idea in the instructions - wrapping `invoke_tool` to catch a bad/missing tool name and either return a graceful message or re-prompt the model, rather than crashing the chain.

**What I learned:** the `invoke_tool` function and the `prompt`/`chain` composition genuinely don't care what the tools *do* - the entire exercise of "porting" this pattern to new tools was really just (1) writing good docstrings, since those become the tool descriptions the model sees verbatim via `render_text_description`, and (2) making sure argument names in the docstring/type hints are unambiguous enough for a non-tool-tuned model like `phi3` to map free text onto them correctly. The few-shot example I added to `system_prompt` (showing the exact JSON for a `count_words` call) is the same few-shot prompting technique from the earlier M-Shots lab, applied here specifically to stabilize the *format* of the tool-call JSON rather than the content of a text answer.